# Demo: Building a Context-Aware E‑Bike Support Assistant with LangChain + Amazon Bedrock

## The business problem

Imagine an e-bike company has a support assistant. A customer may say:

> “I have a CityRide X2. The battery drops from 60% to 20% in about 10 minutes.”

Then the same customer asks:

> “Is that normal?”  
> “What should I check first?”  
> “What model did I tell you I have?”

A foundation model can answer each question, but the **application** must manage prompts, sequencing, and conversation history.

This notebook demonstrates why an open-source framework such as **LangChain** becomes useful as the application grows.

### What we will build

We will evolve the same use case through four stages:

1. Call Amazon Bedrock directly with Boto3.
2. Use LangChain as an abstraction over Amazon Bedrock.
3. Build a reusable prompt + chain for support responses.
4. Add persistent conversation memory with Amazon DynamoDB.

At the end, a cleanup function removes the DynamoDB table created by this demo.


## Architecture

```text
Customer
   |
   v
Notebook / Application
   |
   +--> LangChain Prompt Template
   |       |
   |       v
   +--> Conversation History <----> DynamoDB
   |       |
   |       v
   +--> Amazon Bedrock Converse API
           |
           v
      Foundation Model
```

The important idea is that **Amazon Bedrock provides the model inference service**, while **LangChain helps organize common application-level patterns** such as model wrappers, prompts, chains, and memory.


## 1. Install dependencies

Run this cell once.

> The notebook assumes AWS credentials are already configured and that the identity has permission to invoke the selected Bedrock model and create/read/write/delete the demo DynamoDB table.


In [ ]:
%pip install -qU boto3 langchain langchain-aws langchain-community

## 2. Configuration and AWS permissions

Use the first configuration code cell for an AWS profile/SSO session or enter instructor-provided temporary credentials through hidden prompts. The notebook never stores credentials in its source.

The learner execution role needs `bedrock:InvokeModel` for the chosen model and DynamoDB `CreateTable`, `DescribeTable`, `GetItem`, `PutItem`, `Query`, and `DeleteTable` for `EBikeSupportSessionTable`. Restrict permissions to the selected Region/model and table; see `setup/bedrock_course_roles.md` in the course repository.

Use a model available in your AWS Region and account. If necessary, change `MODEL_ID` to an enabled model or inference profile.


In [ ]:
import getpass
import os
import boto3

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
AWS_AUTH_METHOD = "profile"  # Change to "keys" only when a profile is unavailable.

if AWS_AUTH_METHOD == "profile":
    AWS_PROFILE_NAME = os.getenv("AWS_PROFILE") or None
    aws_session = boto3.Session(profile_name=AWS_PROFILE_NAME, region_name=AWS_REGION)
    boto3.setup_default_session(profile_name=AWS_PROFILE_NAME, region_name=AWS_REGION)
    print("Using AWS profile/default credential chain.")
elif AWS_AUTH_METHOD == "keys":
    print("Enter temporary credentials only. Inputs are hidden and are not saved in this notebook.")
    access_key_id = getpass.getpass("AWS access key ID: ")
    secret_access_key = getpass.getpass("AWS secret access key: ")
    session_token = getpass.getpass("AWS session token (leave blank if not applicable): ")
    credential_args = {
        "aws_access_key_id": access_key_id,
        "aws_secret_access_key": secret_access_key,
        "region_name": AWS_REGION,
    }
    if session_token:
        credential_args["aws_session_token"] = session_token
    aws_session = boto3.Session(**credential_args)
    boto3.setup_default_session(**credential_args)
    del access_key_id, secret_access_key, session_token, credential_args
    print("Using temporary credentials for this kernel session.")
else:
    raise ValueError('AWS_AUTH_METHOD must be "profile" or "keys".')

import uuid
from botocore.exceptions import ClientError

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.amazon.nova-lite-v1:0")
TABLE_NAME = "EBikeSupportSessionTable"

bedrock_runtime = aws_session.client("bedrock-runtime", region_name=AWS_REGION)
dynamodb = aws_session.resource("dynamodb", region_name=AWS_REGION)

print("Region:", AWS_REGION)
print("Model :", MODEL_ID)
print("Table :", TABLE_NAME)


# Part A — What happens without an open-source framework?

First, call Amazon Bedrock directly.

For a single request this is simple and efficient. The extra application work becomes more visible when we need reusable prompts, multi-step processing, or conversation history.


In [ ]:
def ask_bedrock_direct(user_text):
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [{"text": user_text}]
            }
        ],
        inferenceConfig={
            "maxTokens": 500,
            "temperature": 0.2,
            "topP": 0.9
        }
    )

    return response["output"]["message"]["content"][0]["text"]

answer = ask_bedrock_direct(
    "I own a CityRide X2 e-bike. The battery drops from 60% to 20% "
    "after about 10 minutes. Give me three safe troubleshooting checks."
)

print(answer)


### Teaching point

The direct API worked.

Now ask a follow-up **without sending the earlier message again**.


In [ ]:
follow_up = ask_bedrock_direct(
    "What model did I tell you I have?"
)

print(follow_up)


### Why this happens

The model invocation is stateless from the application's point of view. The second request does not automatically contain the first request.

Without a framework, the application would need to:

- maintain a message list for every user/session,
- reconstruct that list on every request,
- keep prompt formatting consistent,
- decide how much history to retain,
- persist history if the app runs across multiple processes or Lambda invocations,
- parse and pass data between steps if the workflow grows.

None of this is impossible with Boto3. The question is whether you want to build and maintain those abstractions yourself.


# Part B — Put LangChain in front of Amazon Bedrock

LangChain provides a standard model interface. We still use an Amazon Bedrock model; LangChain is an application framework around the interaction.


In [ ]:
from langchain_aws import ChatBedrockConverse

model = ChatBedrockConverse(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.2,
    max_tokens=500,
)

response = model.invoke(
    "Explain regenerative braking on an e-bike in two sentences for a new rider."
)

print(response.content)


## What did LangChain solve here?

For this one call, not much business complexity has been removed yet—and that is an important point.

The value becomes clearer when we start composing reusable application components.


# Part C — Reusable prompt template

A support application should not rebuild a long prompt string everywhere in the code.

We will create one prompt that standardizes:

- the assistant's role,
- safety behavior,
- response structure,
- customer and bike information.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

support_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an e-bike technical support assistant.

Follow these rules:
1. Give safe, non-destructive troubleshooting steps.
2. Do not tell the customer to open a battery pack or perform hazardous electrical repairs.
3. If the symptom could indicate battery damage, overheating, smoke, swelling, or fire risk,
   tell the customer to stop using/charging the bike and seek qualified service.
4. Keep the response concise.

Return:
- Likely category
- First checks
- When to escalate"""
    ),
    (
        "human",
        """Bike model: {bike_model}
Customer issue: {issue}"""
    )
])

prompt_value = support_prompt.invoke({
    "bike_model": "CityRide X2",
    "issue": "Battery falls from 60% to 20% after about 10 minutes."
})

print(prompt_value)


# Part D — Build a chain

A **chain** connects components so that the output of one component becomes the input to the next.

Here the flow is:

```text
input dictionary -> prompt template -> Bedrock model -> text parser
```


In [ ]:
from langchain_core.output_parsers import StrOutputParser

support_chain = support_prompt | model | StrOutputParser()

result = support_chain.invoke({
    "bike_model": "CityRide X2",
    "issue": "Battery falls from 60% to 20% after about 10 minutes."
})

print(result)


## Why the chain matters

Application code can now call one object:

```python
support_chain.invoke({...})
```

The caller does not need to know how the prompt is constructed or how the model response is converted into text.

That becomes useful when the workflow grows into:

```text
customer message
      |
      v
classify issue
      |
      v
generate troubleshooting response
      |
      v
store / retrieve conversation context
```


# Part E — A simple two-step workflow

Let's make the value of composition more visible.

Step 1 classifies the support request.  
Step 2 generates guidance using the classification.


In [ ]:
classification_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Classify the e-bike support request into exactly one category: "
        "BATTERY, MOTOR, BRAKES, DISPLAY, CONNECTIVITY, or OTHER."
    ),
    ("human", "{issue}")
])

classification_chain = classification_prompt | model | StrOutputParser()

issue = "The battery falls from 60% to 20% after about 10 minutes."
category = classification_chain.invoke({"issue": issue}).strip()

print("Detected category:", category)


In [ ]:
triage_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an e-bike support assistant.
The request has already been classified.

Give:
1. A one-sentence assessment
2. Three safe checks
3. One escalation condition

Do not recommend opening or repairing a battery pack."""
    ),
    (
        "human",
        """Category: {category}
Bike model: {bike_model}
Issue: {issue}"""
    )
])

triage_chain = triage_prompt | model | StrOutputParser()

triage = triage_chain.invoke({
    "category": category,
    "bike_model": "CityRide X2",
    "issue": issue
})

print(triage)


# Part F — The real conversational problem: memory

Suppose the customer has a longer conversation:

```text
Customer: My name is Ravi. I have a CityRide X2.
Customer: The battery drops quickly.
Customer: What model did I say I have?
Customer: Should I keep charging it?
```

In a web application or AWS Lambda architecture, separate requests may run in separate executions.

We therefore need **persistent session history**, not just a Python list living in one notebook process.

For this demo we will store chat history in Amazon DynamoDB.


## 3. Create the DynamoDB session table

LangChain's DynamoDB chat history expects a table keyed by `SessionId`.

This cell creates the table only if it does not already exist.


In [ ]:
def create_session_table(table_name=TABLE_NAME):
    client = boto3.client("dynamodb", region_name=AWS_REGION)

    try:
        client.describe_table(TableName=table_name)
        print(f"Table already exists: {table_name}")
        return dynamodb.Table(table_name)
    except client.exceptions.ResourceNotFoundException:
        pass

    table = dynamodb.create_table(
        TableName=table_name,
        KeySchema=[
            {"AttributeName": "SessionId", "KeyType": "HASH"}
        ],
        AttributeDefinitions=[
            {"AttributeName": "SessionId", "AttributeType": "S"}
        ],
        BillingMode="PAY_PER_REQUEST"
    )

    print(f"Creating table: {table_name}")
    table.wait_until_exists()
    print("Table is ready.")
    return table

session_table = create_session_table()


# Part G — LangChain prompt + model + DynamoDB history

This is the key framework pattern from the module:

```text
Prompt Template
      +
Conversation History
      +
Bedrock Model
      =
Context-aware chain
```

The same `session_id` retrieves the same user's previous messages.


In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import DynamoDBChatMessageHistory

chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an e-bike customer support assistant.
Remember relevant information the customer has already provided in this session.
Give concise and safe guidance.
Never recommend opening or repairing an e-bike battery pack."""
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

chat_chain = chat_prompt | model

def get_session_history(session_id: str):
    return DynamoDBChatMessageHistory(
        table_name=TABLE_NAME,
        session_id=session_id,
        boto3_session=aws_session,
    )

chat_with_history = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

print("Context-aware chain created.")


## 4. Start a customer session


In [ ]:
SESSION_ID = "customer-" + str(uuid.uuid4())[:8]
config = {"configurable": {"session_id": SESSION_ID}}

print("Session ID:", SESSION_ID)


In [ ]:
response1 = chat_with_history.invoke(
    {
        "question": (
            "Hi, my name is Ravi. I have a CityRide X2 e-bike. "
            "Its battery drops from 60% to 20% after about 10 minutes."
        )
    },
    config=config
)

print(response1.content)


Now ask a follow-up without repeating the bike model.


In [ ]:
response2 = chat_with_history.invoke(
    {"question": "What model did I tell you I have?"},
    config=config
)

print(response2.content)


And ask a contextual troubleshooting question.


In [ ]:
response3 = chat_with_history.invoke(
    {"question": "Based on the battery issue I described, what should I check first?"},
    config=config
)

print(response3.content)


# Part H — Prove that the memory is really in DynamoDB

We can inspect the stored session item.


In [ ]:
table = dynamodb.Table(TABLE_NAME)

item = table.get_item(
    Key={"SessionId": SESSION_ID}
).get("Item")

item


## Simulate a new application/Lambda invocation

We create a new history object using the same session ID.

The previous Python responses are not manually passed into this call. The history can be reconstructed from DynamoDB.


In [ ]:
new_history_object = DynamoDBChatMessageHistory(
    table_name=TABLE_NAME,
    session_id=SESSION_ID,
    boto3_session=aws_session,
)

print("Messages loaded from DynamoDB:", len(new_history_object.messages))

for message in new_history_object.messages:
    print(type(message).__name__, ":", message.content[:120])


In [ ]:
response4 = chat_with_history.invoke(
    {"question": "What was my name, and what bike model do I own?"},
    config={"configurable": {"session_id": SESSION_ID}}
)

print(response4.content)


# What problem did the open-source framework solve?

The framework did **not** replace Amazon Bedrock.

Amazon Bedrock still performs model inference.

LangChain helped us organize common generative-AI application concerns:

| Application problem | Without framework | LangChain approach |
|---|---|---|
| Calling a model | Build provider-specific request/response code | Standard chat model interface |
| Prompt reuse | Build strings manually | `ChatPromptTemplate` |
| Multi-step workflow | Manually pass outputs between functions | Chains / LCEL |
| Conversation history | Manually maintain message arrays | `RunnableWithMessageHistory` |
| Persistent chat history | Write custom storage plumbing | `DynamoDBChatMessageHistory` |
| Swap/compose components | Refactor application glue code | Common component interfaces |

### When direct Bedrock APIs may be enough

Use the direct API when the application is small, the workflow is simple, and you want minimal abstraction.

### When a framework becomes attractive

A framework becomes more useful when the application needs combinations of:

- reusable prompts,
- multiple model/tool steps,
- conversation history,
- persistent memory,
- retrieval,
- agents/tools,
- output parsing,
- swappable components.

The tradeoff is that a third-party framework adds another dependency and abstraction layer that your team must understand and maintain.


# Instructor demo flow

A useful way to teach this notebook is to reveal the need for each feature rather than introducing LangChain first.

### Act 1 — “Bedrock already works. Why do I need LangChain?”

Run the direct Boto3 example.

Ask the class:

> “If this works, why add another framework?”

Then run:

```text
What model did I tell you I have?
```

The missing context creates the problem.

### Act 2 — “Reusable application components”

Introduce:

```text
Model wrapper -> PromptTemplate -> Chain
```

Explain that the benefit is not fewer lines in every tiny example. It is having reusable abstractions as the application grows.

### Act 3 — “Now make it conversational”

Add `RunnableWithMessageHistory`.

Use the customer's name and bike model, then ask follow-up questions.

### Act 4 — “What happens when Lambda finishes?”

Explain that in-memory state is not sufficient for persistent sessions.

Add DynamoDB chat history and rerun the same session.

### Final question to students

> “Would you use direct Bedrock APIs or LangChain for a one-prompt batch summarization job?”

Direct APIs may be perfectly reasonable.

Then ask:

> “What about a support assistant with prompt templates, RAG, memory, tools, and multiple workflow steps?”

That is the point where an orchestration framework can reduce application plumbing.


# Optional extension ideas

Once students understand the core pattern, the same assistant can evolve into:

```text
Customer question
      |
      v
Conversation memory
      |
      +--> Retrieve product/manual information
      |
      +--> Classify support issue
      |
      +--> Call warranty/order tools
      |
      v
Context-aware answer
```

This naturally leads into RAG, retrievers, tools, and agents without changing the original business story.


# Cleanup

Run this cell after the demo to delete the DynamoDB table created by this notebook.

> This does not delete or modify Amazon Bedrock models. The notebook creates only the DynamoDB table.


In [ ]:
def cleanup_demo_resources(table_name=TABLE_NAME):
    client = boto3.client("dynamodb", region_name=AWS_REGION)

    try:
        client.describe_table(TableName=table_name)
    except client.exceptions.ResourceNotFoundException:
        print(f"No table to delete: {table_name}")
        return

    print(f"Deleting DynamoDB table: {table_name}")
    client.delete_table(TableName=table_name)

    waiter = client.get_waiter("table_not_exists")
    waiter.wait(TableName=table_name)

    print("Cleanup complete.")

# Uncomment after finishing the demo:
# cleanup_demo_resources()


# Key takeaway

**Amazon Bedrock gives us access to foundation models.**

**An open-source framework such as LangChain helps us compose the surrounding application behavior.**

For this demo, the progression was:

```text
Direct model call
      ↓
Model abstraction
      ↓
Reusable prompt
      ↓
Chain
      ↓
Conversation history
      ↓
Persistent DynamoDB memory
      ↓
Context-aware application
```

That is the problem open-source orchestration frameworks are helping solve.
